# SVC 후속 실험 시작 템플릿
현재 선정 모델을 검증한 후 사전등록한 후보만 실행한다. 예시 조건은 **계획이며 실행 결과가 아니다**. 개발 1,280장과 공식 Validation 19,200장만 평가. Original Test/교수님 Test 금지. 후보 실행은 기본 False. 환경 설정 및 입력은 공통 README 참조. 결과는 로컬 폴더에 저장하며 선택 모델을 바꾸지 않는다.

In [ ]:
from pathlib import Path
import sys, time, json, traceback
from datetime import datetime
import numpy as np
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # edit
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("SVC_next_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # SVC only: set path of verified C2_G2 model.joblib
state=prepare("SVC",INPUT_DIR,OUTPUT_DIR,mode="followup",current_svc=CURRENT_SVC)
print("Baseline reproduced; local output:",OUTPUT_DIR)

## 후보 학습 및 집계 평가
아래 조건은 예시. 실행 전 계획을 확정하고 ID를 고유하게 부여한다. matched Batch1 시간은 별도로 측정하여 기록한다.

In [ ]:
RUN_CANDIDATE=False # enable only after preregistering the condition
EXPERIMENT_ID="FT-NEXT-SVC-001"
CONFIG={"C":2.0,"gamma_multiplier":2.0,"handmade_weight":1.0,"target_weight":2.0,"seed":42,"training_scope":"approved training only","condition":"proposed full combination"}
if RUN_CANDIDATE:
    from sklearn.svm import SVC
    log=OUTPUT_DIR/(EXPERIMENT_ID+"_fit.log")
    try:
        (OUTPUT_DIR/(EXPERIMENT_ID+"_plan.json")).write_text(json.dumps(CONFIG,indent=2))
        t=time.perf_counter()
        X,y,meta=state["fb"].train_features("SVC_Combined",state["off"],state["wr"])
        assert X.shape==(82080,1108)
        weights=np.ones(len(y))
        weights[76800:]*=CONFIG["handmade_weight"]
        weights[76800:][np.isin(y[76800:],[11,13,12])]*=CONFIG["target_weight"]
        gamma=float(CONFIG["gamma_multiplier"]/(X.shape[1]*X.var()))
        candidate=SVC(C=CONFIG["C"],gamma=gamma,cache_size=200)
        candidate.fit(X,y,sample_weight=weights)
        fit_s=time.perf_counter()-t
        log.write_text(f"FIT COMPLETED {fit_s}s\n")
        print(evaluate_followup(state,"SVC",EXPERIMENT_ID,candidate.predict,CONFIG,fit_s))
    except Exception:
        log.write_text(traceback.format_exc())
        raise
else:
    print("No candidate executed. Set RUN_CANDIDATE only after approval.")

## 기록
실측 metrics.csv/metrics.json, CM, 실행 로그, 환경과 계획을 보존한다. 별도 정식 시간 평가 및 팀 승인 없이는 최종 모델 교체 금지.